## LangChain chains

In [1]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
%pip install -qU langchain langchain-core langchain-openai langchain-classic python-dotenv
# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notatnikowi na dostęp.
# Odkomentuj te linie, aby wczytać klucz z Colab Secrets do bieżącej sesji:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Lokalnie umieść OPENAI_API_KEY=... w pliku .env; nie commituj klucza.

Note: you may need to restart the kernel to use updated packages.


### Prosty chain: prompt → model → wynik

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Jesteś ekspertem programowania w Pythonie i geniuszem w dziedzinie AI."),
    ("user", "Napisz kod związany z tematem {topic}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = prompt | llm | StrOutputParser()
result = chain.invoke({'topic': 'AGI'})


In [3]:
print(result)

Tworzenie sztucznej inteligencji ogólnej (AGI) to bardzo ambitne zadanie, które wykracza poza możliwości pojedynczego skryptu w Pythonie. AGI odnosi się do systemów AI, które mogą wykonywać dowolne zadanie intelektualne, które potrafi wykonać człowiek. Niemniej jednak, mogę zaprezentować prosty przykład, który ilustruje podstawowe koncepcje związane z uczeniem maszynowym i przetwarzaniem języka naturalnego, które są kluczowe w rozwoju bardziej zaawansowanych systemów AI.

Poniżej znajduje się przykład prostego modelu klasyfikacji tekstu przy użyciu biblioteki `scikit-learn` oraz `nltk`. Model ten może być użyty do klasyfikacji sentymentu w tekstach, co jest jednym z kroków w kierunku bardziej zaawansowanej AI.

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import nltk
from nltk.corpus

### Sequential chain: dwa modele w sekwencji

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1) Chain: streszczenie (wejście: {text} → wyjście: str)
summary_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Streść poniższy tekst w 1–2 zdaniach po polsku."),
        ("user", "{text}")
    ])
    | llm
    | StrOutputParser()
)

# 2) Adapter: zamień str → {"text": str}, żeby podać do następnego promptu
to_dict = RunnableLambda(lambda s: {"text": s})

# 3) Chain: tłumaczenie streszczenia na francuski (wejście: {text} → wyjście: str)
translate_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Przetłumacz tekst na język francuski."),
        ("user", "{text}")
    ])
    | llm
    | StrOutputParser()
)

sequential_chain = summary_chain | to_dict | translate_chain
input_text = "LangChain umożliwia tworzenie aplikacji AI poprzez łączenie modeli, promptów i narzędzi w spójne pipeline’y."

final_translation = sequential_chain.invoke({"text": input_text})

print(final_translation)


LangChain permet de créer des applications d'IA en intégrant des modèles, des invites et des outils dans des processus organisés.


### Branching chain: jedna odpowiedź, dwa przetworzenia

In [5]:
from langchain_core.runnables import RunnableParallel

# Prompt do streszczenia
prompt_summary = ChatPromptTemplate.from_template("Streść: {text}")

# Prompt do sentymentu
prompt_sentiment = ChatPromptTemplate.from_template("Określ ton wypowiedzi: {text}")

branch_chain = RunnableParallel(
    summary=(prompt_summary | llm | StrOutputParser()),
    sentiment=(prompt_sentiment | llm | StrOutputParser())
)

text = "Jestem bardzo zadowolony z tego kursu, nauczyłem się dużo o LangChain! Teraz wiem, że LangChain umożliwia tworzenie aplikacji AI poprzez łączenie modeli, promptów i narzędzi w spójne pipeline’y."
result = branch_chain.invoke({"text": text})

print(result)


{'summary': 'Uczestnik kursu jest bardzo zadowolony z nauki o LangChain, który pozwala na tworzenie aplikacji AI poprzez integrację modeli, promptów i narzędzi w spójne pipeline’y.', 'sentiment': 'Ton wypowiedzi jest pozytywny i entuzjastyczny. Osoba wyraża zadowolenie z kursu oraz podkreśla zdobyte umiejętności i wiedzę na temat LangChain, co sugeruje, że jest zmotywowana i zaintrygowana tematem.'}
